Title: tas_julia_2005_2024.ipynb

Purpose: This notebook is used to calculate the global warming level for the period of 2005-2024. It uses code from this notebook: https://everval.github.io/Global-Temperature-Anomalies/#introduction

Author: Onno Nennecke on 09.02.2026 Modified: 03.11.2026

Input data: 

- Next to the data downloaded inside of the notebook, Original Data can also be found here: https://www.metoffice.gov.uk/hadobs/hadcrut5/data/HadCRUT.5.1.0.0/download.html


In [1]:
# Install necessary packages
# This code installs the packages if they are not already installed.
using Pkg
Pkg.add(["Plots", "Dates", "CSV", "DataFrames", "HTTP"])

# Load necessary packages
using Pkg
Pkg.activate(pwd())
using Dates, CSV, DataFrames, HTTP, Statistics


   Resolving package versions...
     Project No packages added to or removed from `~/Code/01_Annual_Temperatures/Project.toml`
    Manifest No packages added to or removed from `~/Code/01_Annual_Temperatures/Manifest.toml`
  Activating project at `~/Code/01_Annual_Temperatures`


In [2]:
# Function to convert wide format data to long format

function longseries(data)
    height = size(data, 1) # Number of rows, equivalent to the number of years
    last_row = 12 - count(ismissing, data[end, 2:13]) # Number of non-missing months in the last year

    many = (height - 1) * 12 + last_row # Total number of months in the long format
    long = zeros(many, 1) # Long format array

    for ii = 1:(height-1) # Loop through all years except the last one
        for jj = 1:12 # Loop through all months
            long[(ii-1)*12+jj] = data[ii, jj+1]
        end
    end

    for jj = 1:last_row # Loop through the last year
        long[(height-1)*12+jj] = data[height, jj+1]
    end

    return long
end

longseries (generic function with 1 method)

## Hadcrut

In [3]:
# Download the HadCRUT temperature data 
# URL of the HadCRUT5 global monthly average CSV
hurl = "https://www.metoffice.gov.uk/hadobs/hadcrut5/data/HadCRUT.5.1.0.0/analysis/diagnostics/HadCRUT.5.1.0.0.analysis.summary_series.global.monthly.csv"

# Local filename to save
hfilename = "/climca/people/onennecke/global_temp_data/HadCRUT5_global_monthly_average.csv"

# If the file does not exist, download it
if !isfile(hfilename)
    println("Downloading HadCRUT5 data...")
    HTTP.download(hurl, hfilename)

    rawhadcrut = CSV.read(hfilename, DataFrame);

    rename!(rawhadcrut, :Time => :Date);
    rename!(rawhadcrut, :"Anomaly (deg C)" => :RawTemperature);

    hadcrut = rawhadcrut[!, [:Date, :RawTemperature]];

    oldbase = mean(hadcrut[(hadcrut.Date.>=Date(1850, 1, 1)).&(hadcrut.Date.<Date(1900, 1, 1)), :RawTemperature]);
    hadcrut[!, :Temp] = hadcrut[!, :RawTemperature] .- oldbase;

    CSV.write(hfilename, hadcrut);
end

hadcrut = CSV.read(hfilename, DataFrame);

println(first(hadcrut, 5));

5×3 DataFrame
 Row │ Date        RawTemperature  Temp         
     │ Date        Float64         Float64      
─────┼──────────────────────────────────────────
   1 │ 1850-01-01       -0.733695  -0.37379
   2 │ 1850-02-01       -0.360436  -0.000531503
   3 │ 1850-03-01       -0.627131  -0.267227
   4 │ 1850-04-01       -0.605297  -0.245392
   5 │ 1850-05-01       -0.531517  -0.171612


In [4]:
# Calcualte mean temperature for period 1995-2014 and 2005-2024
mean_1995_2014_hc = mean(hadcrut[(hadcrut.Date.>=Date(1995, 1, 1)).&(hadcrut.Date.<Date(2015, 1, 1)), :Temp]);
mean_2005_2024_hc = mean(hadcrut[(hadcrut.Date.>=Date(2005, 1, 1)).&(hadcrut.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014_hc, " °C");
println("Mean temperature anomaly (2005-2024): ", mean_2005_2024_hc, " °C");

Mean temperature anomaly (1995-2014): 0.8679792232733333 °C
Mean temperature anomaly (2005-2024): 1.1024848617316665 °C


### GISTEMP

In [5]:
# Download the GISTEMP temperature data 
# URL of the GISTEMP global monthly average CSV
gurl = "https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts%2BdSST.csv"
# Local filename to save
gfilename = "/climca/people/onennecke/global_temp_data/GISTEMP_global_monthly_average.csv"
# Download the file

# If the file does not exist, download it
if !isfile(gfilename)
    println("Downloading GISTEMP data...")
    HTTP.download(gurl, gfilename)

    longgistemp = CSV.read(gfilename, DataFrame, header=2, missingstring=["***"]);
    gistemp = longseries(longgistemp)[:];
    Tt = length(gistemp) - 1;

    start = Date(1880, 1, 1); # Start date of the dataset
    fin = start + Month(Tt); # End date of the dataset
    fechas = collect(start:Month(1):fin); # Create a Date array

    gistemp = DataFrame(:Date=>fechas, :RawTemp=>gistemp);

    oldbase = mean(gistemp[(gistemp.Date.>=Date(1880, 1, 1)).&(gistemp.Date.<Date(1900, 1, 1)), :RawTemp]);
    gistemp[!, :Temp] = gistemp[!, :RawTemp] .- oldbase .+ 0.038; # Adjust for pre-1880 data

    CSV.write(gfilename, gistemp);
end

gistemp = CSV.read(gfilename, DataFrame);

println(first(gistemp, 5));

5×3 DataFrame
 Row │ Date        RawTemp  Temp      
     │ Date        Float64  Float64   
─────┼────────────────────────────────
   1 │ 1880-01-01    -0.19  0.0725417
   2 │ 1880-02-01    -0.25  0.0125417
   3 │ 1880-03-01    -0.09  0.172542
   4 │ 1880-04-01    -0.16  0.102542
   5 │ 1880-05-01    -0.1   0.162542


In [6]:
# Calcualte mean temperature for period 1995-2014 and 2005-2024
mean_1995_2014_gt = mean(gistemp[(gistemp.Date.>=Date(1995, 1, 1)).&(gistemp.Date.<Date(2015, 1, 1)), :Temp]);
mean_2005_2024_gt = mean(gistemp[(gistemp.Date.>=Date(2005, 1, 1)).&(gistemp.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014_gt, " °C");
println("Mean temperature anomaly (2005-2024): ", mean_2005_2024_gt, " °C");

Mean temperature anomaly (1995-2014): 0.8385833333333336 °C
Mean temperature anomaly (2005-2024): 1.083375 °C


### Berkeley Earth

In [7]:
# Download the Berkeley Earth temperature data
# URL of the Berkeley Earth global monthly average CSV
burl = "https://storage.googleapis.com/berkeley-earth-temperature-hr/global/Global_TAVG_monthly.txt"
# Local filename to save
bfilename = "/climca/people/onennecke/global_temp_data/BerkeleyEarth_global_monthly_average.csv"

# If the file does not exist, download it
if !isfile(bfilename)
    println("Downloading Berkeley Earth data...")
    HTTP.download(burl, bfilename)

    rawtemp = CSV.read(bfilename, DataFrame, comment="%", delim=" ", ignorerepeated=true);

    colnames = [:Year, :Month, :Anomaly_Monthly, :Unc_Monthly,
                :Anomaly_Annual, :Unc_Annual, :Anomaly_5yr, :Unc_5yr,
                :Anomaly_10yr, :Unc_10yr, :Anomaly_20yr, :Unc_20yr];
    rename!(rawtemp, colnames);

    rawtemp.Date = Date.(rawtemp.Year, rawtemp.Month, 1); # Create Date column from Year and Month
    rename!(rawtemp, :Anomaly_Monthly => :RawTemperature);

    berkeley = rawtemp[!, [:Date, :RawTemperature]];

    oldbase = mean(rawtemp[(rawtemp.Date.>=Date(1850, 1, 1)).&(rawtemp.Date.<Date(1900, 1, 1)), :RawTemperature]);
    rawtemp[!, :Temp] = rawtemp[!, :RawTemperature] .- oldbase;

    berkeley.Temp = rawtemp.Temp;

    CSV.write(bfilename, berkeley);
end

berkeley = CSV.read(bfilename, DataFrame);
println(first(berkeley, 5));

5×3 DataFrame
 Row │ Date        RawTemperature  Temp       
     │ Date        Float64         Float64    
─────┼────────────────────────────────────────
   1 │ 1850-01-01          -0.542  -0.314937
   2 │ 1850-02-01          -0.287  -0.0599367
   3 │ 1850-03-01          -0.256  -0.0289367
   4 │ 1850-04-01          -0.652  -0.424937
   5 │ 1850-05-01          -0.183   0.0440633


In [8]:
# Calcualte mean temperature for period 1995-2014 and 2005-2024
mean_1995_2014_berkeley = mean(berkeley[(berkeley.Date.>=Date(1995, 1, 1)).&(berkeley.Date.<Date(2015, 1, 1)), :Temp]);
mean_2005_2024_berkeley = mean(berkeley[(berkeley.Date.>=Date(2005, 1, 1)).&(berkeley.Date.<Date(2025, 1, 1)), :Temp]);

println("Mean temperature anomaly (1995-2014): ", mean_1995_2014_berkeley, " °C");
println("Mean temperature anomaly (2005-2024): ", mean_2005_2024_berkeley, " °C");

Mean temperature anomaly (1995-2014): 0.8250966666666668 °C
Mean temperature anomaly (2005-2024): 1.0603466666666665 °C


## Overview

In [9]:
mean_1995_2014_all = mean([mean_1995_2014_hc, mean_1995_2014_gt, mean_1995_2014_berkeley]);
mean_2005_2024_all = mean([mean_2005_2024_hc, mean_2005_2024_gt, mean_2005_2024_berkeley]);

In [10]:
# Print all temperature anomalies together
println("Mean temperature anomalies (1995-2014):");
println("HadCRUT: ", mean_1995_2014_hc, " °C");
println("GISTEMP: ", mean_1995_2014_gt, " °C");
println("Berkeley Earth: ", mean_1995_2014_berkeley, " °C");
println("Mean: ", mean_1995_2014_all, " °C");

println("\nMean temperature anomalies (2005-2024):");
println("HadCRUT: ", mean_2005_2024_hc, " °C");
println("GISTEMP: ", mean_2005_2024_gt, " °C");
println("Berkeley Earth: ", mean_2005_2024_berkeley, " °C");
println("Mean: ", mean_2005_2024_all, " °C");

Mean temperature anomalies (1995-2014):
HadCRUT: 0.8679792232733333 °C
GISTEMP: 0.8385833333333336 °C
Berkeley Earth: 0.8250966666666668 °C
Mean: 0.8438864077577778 °C

Mean temperature anomalies (2005-2024):
HadCRUT: 1.1024848617316665 °C
GISTEMP: 1.083375 °C
Berkeley Earth: 1.0603466666666665 °C
Mean: 1.0820688427994443 °C


In [11]:
# Save the results to a CSV file
results = DataFrame(
    Dataset = ["HadCRUT", "GISTEMP", "Berkeley Earth", "Mean"],
    Mean_1995_2014 = [mean_1995_2014_hc, mean_1995_2014_gt, mean_1995_2014_berkeley, mean_1995_2014_all],
    Mean_2005_2024 = [mean_2005_2024_hc, mean_2005_2024_gt, mean_2005_2024_berkeley, mean_2005_2024_all]
)

CSV.write("/climca/people/onennecke/global_temp_data/mean_temperature_anomalies_all_datasets.csv", results);

In [12]:
results

Row,Dataset,Mean_1995_2014,Mean_2005_2024
,String,Float64,Float64
1,HadCRUT,0.867979,1.10248
2,GISTEMP,0.838583,1.08337
3,Berkeley Earth,0.825097,1.06035
4,Mean,0.843886,1.08207
